# Figure 3C — S100P and CIITA expression in LUAD epithelial cells

**Goal:** Characterize the relationship between S100P and CIITA expression
in paired normal adjacent and primary tumor epithelial cells from the
Salcher LUAD atlas. Two complementary analyses are shown:

1. Simple paired comparison — S100P and CIITA expression in normal adjacent
   vs primary tumor epithelial cells, showing that CIITA is selectively
   suppressed in tumor.

2. Within-tumor S100P stratification — among CIITA-expressing cells, does
   S100P status predict lower CIITA expression? Tests whether the S100P–MHC II
   axis operates at the level of the CIITA transcription factor.

**Input:** `luad_epithelial_harmony.h5ad` (Salcher LUAD epithelial subset).

**Output:** Figure 3C panels saved to `outputs/figures/figure3/`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad
import scanpy as sc
import scipy.sparse as sp
from scipy.stats import wilcoxon, mannwhitneyu
from pathlib import Path
import yaml

sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])
fig_dir   = repo_root / cfg['outputs']['figures']
table_dir = repo_root / cfg['outputs']['tables']

fig_out   = fig_dir   / 'figure3'
table_out = table_dir / 'figure3'

fig_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

## Load data

Salcher LUAD epithelial subset with Harmony-corrected embedding. Restricted
to paired donors with both normal adjacent and primary tumor samples.

In [ ]:
epithelial_path = repo_root / cfg['outputs']['processed'] / 'luad_epithelial_harmony.h5ad'
epithelial = ad.read_h5ad(epithelial_path)

# filter to LUAD paired donors
luad = epithelial[
    epithelial.obs['origin'].isin(['tumor_primary', 'normal_adjacent'])
].copy()

paired_donors = (
    luad.obs.groupby('donor_id')['origin'].nunique()
)
paired_donors = paired_donors[paired_donors == 2].index
luad_paired = luad[luad.obs['donor_id'].isin(paired_donors)].copy()

print(f"Paired LUAD epithelial cells: {luad_paired.n_obs:,}")
print(f"Paired donors: {len(paired_donors)}")
print(luad_paired.obs['origin'].value_counts())

In [ ]:
# resolve Ensembl IDs for S100P and CIITA
s100p_ens = luad_paired.var.loc[luad_paired.var['feature_name'] == 'S100P'].index[0]
ciita_ens  = luad_paired.var.loc[luad_paired.var['feature_name'] == 'CIITA'].index[0]
print(f"S100P: {s100p_ens}")
print(f"CIITA: {ciita_ens}")

## Figure 3C part 1 — paired expression comparison

Sample-level mean expression of S100P and CIITA in normal adjacent vs
primary tumor epithelial cells. Each line connects the same donor's
normal and tumor sample. Wilcoxon signed-rank test on paired differences.

In [ ]:
from ceiba.plot_utils import (
    plot_genes_paired_luad,
    plot_genes_paired_luad_percent_detected,
    plot_genes_pct_expressing_luad,
    plot_scrna_group_comparison,
    plot_celltype_comparison_luad,
)
from ceiba.stats_utils import (
    ciita_expr_by_s100p_strata_per_sample,
    ciita_expr_cell_level_tests,
)

In [ ]:
#from ceiba.plot_utils import plot_genes_paired_luad

plot_genes_paired_luad(
    luad_paired,
    genes=['S100P', 'CIITA'],
    figsize_per_gene=(5, 5),
    save_path=fig_out / 'fig3c_s100p_ciita_paired.pdf',
)
plt.show()

## Figure 3C part 2 — S100P stratification of CIITA expression

Among CIITA-expressing epithelial cells, mean CIITA expression is compared
between S100P+ and S100P- cells within each sample. The within-sample delta
(S100P+ minus S100P-) is tested against zero by Wilcoxon signed-rank test,
run separately for normal adjacent and primary tumor. A consistently negative
delta in primary tumor indicates that S100P+ tumor cells express less CIITA
even among cells that detectably transcribe it — implicating the CIITA
transcription factor in the S100P–MHC II axis.

In [ ]:
from ceiba.stats_utils import ciita_expr_by_s100p_strata_per_sample

wide = ciita_expr_by_s100p_strata_per_sample(luad_paired, agg='mean_detected_only')

# print Wilcoxon results
for origin in ['normal_adjacent', 'tumor_primary']:
    sub = wide[wide['origin'] == origin].dropna(subset=['delta_pos_minus_neg'])
    if len(sub) == 0:
        print(f'{origin}: no paired strata')
        continue
    stat, p = wilcoxon(sub['delta_pos_minus_neg'])
    print(
        f'{origin}  n_samples={len(sub)}'
        f'  Wilcoxon(delta pos-neg) p={p:.3e}'
        f'  median_delta={np.nanmedian(sub["delta_pos_minus_neg"]):.3g}'
    )

wide.to_csv(table_out / 'ciita_s100p_strata.csv', index=False)

In [ ]:
from ceiba.stats_utils import ciita_expr_cell_level_tests

cell_results = ciita_expr_cell_level_tests(
    luad_paired, restrict_ciita_pos=True
)
print(cell_results.to_string(index=False))
cell_results.to_csv(table_out / 'ciita_s100p_cell_level_test.csv', index=False)

In [ ]:
def plot_ciita_s100p_paired(wide, figsize=(8, 3.8), dpi=150):
    """
    Two-panel paired dot + delta figure comparing CIITA expression in S100P+
    vs S100P- epithelial cells, separately for normal adjacent and tumor.

    Left panel of each pair: per-sample paired dot plot (S100P- vs S100P+).
    Right panel: within-sample delta (S100P+ minus S100P-) with Wilcoxon test.

    Parameters
    ----------
    wide : pd.DataFrame
        Output of ciita_expr_by_s100p_strata_per_sample.
    figsize : tuple
        Figure dimensions in inches.
    dpi : int
        Figure resolution.

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    origins = ['normal_adjacent', 'tumor_primary']
    labels  = ['Normal adjacent', 'Tumor']


    COLOR_NEG  = '#C4A882'
    COLOR_POS  = '#7B2D8B'
    
    # origin colors
    ORIGIN_COLORS = {
        'normal_adjacent': 'tab:grey',
        'tumor_primary':   'tab:red',
    }
    
    ALPHA_LINE = 0.35
    DOT_SIZE   = 4

    fig = plt.figure(figsize=figsize, dpi=dpi)
    gs  = fig.add_gridspec(
        2, 4,
        width_ratios=[1.6, 0.9, 1.6, 0.9],
        height_ratios=[1, 1],
        left=0.1, right=0.97, top=0.92, bottom=0.08,
        wspace=0.25, hspace=0.35,
    )
    
    ax_slots = [
        fig.add_subplot(gs[0, 0]),  # normal adjacent paired
        fig.add_subplot(gs[1, 0]),  # normal adjacent delta
        fig.add_subplot(gs[0, 2]),  # tumor paired
        fig.add_subplot(gs[1, 2]),  # tumor delta
    ]

    for i, (origin, label) in enumerate(zip(origins, labels)):
        ax_paired = ax_slots[i * 2]      # 0 for normal, 2 for tumor
        ax_delta  = ax_slots[i * 2 + 1]  # 1 for normal, 3 for tumor

        color = ORIGIN_COLORS[origin]

        sub = wide[wide['origin'] == origin].dropna(
            subset=['CIITA_mean_S100Pneg', 'CIITA_mean_S100Ppos', 'delta_pos_minus_neg']
        )
        deltas   = sub['delta_pos_minus_neg'].values
        stat, p  = wilcoxon(deltas)
        s        = sig_label(p)
        med      = np.median(deltas)

        # paired dot plot
        rng    = np.random.default_rng(42 + i)
        jitter = rng.uniform(-0.08, 0.08, len(sub))

        for xn, xp, yn, yp in zip(
            jitter, 1 + jitter,
            sub['CIITA_mean_S100Pneg'],
            sub['CIITA_mean_S100Ppos'],
        ):
            ax_paired.plot([xn, xp], [yn, yp],
                           color=color, lw=0.6, alpha=ALPHA_LINE, zorder=1)

        ax_paired.scatter(jitter,     sub['CIITA_mean_S100Pneg'],
                          color=color, s=DOT_SIZE, zorder=3)
        ax_paired.scatter(1 + jitter, sub['CIITA_mean_S100Ppos'],
                          color=color, s=DOT_SIZE, zorder=3)

        ax_paired.set_xticks([0, 1])
        ax_paired.set_xticklabels(['S100P-', 'S100P+'], fontsize=8)
        ax_paired.set_xlim(-0.4, 1.4)
        ax_paired.set_ylabel('Mean CIITA (CIITA+ cells)', fontsize=7.5)
        ax_paired.set_title(label, fontsize=9, fontweight='bold', pad=6)
        ax_paired.spines[['top', 'right']].set_visible(False)
        ax_paired.tick_params(labelsize=7)

        # delta strip plot
        rng2 = np.random.default_rng(99 + i)
        jit2 = rng2.uniform(-0.18, 0.18, len(deltas))
        ymin = min(deltas) - np.ptp(deltas) * 0.08
        ymax = max(deltas) + np.ptp(deltas) * 0.45

        ax_delta.set_ylim(ymin, ymax)
        ax_delta.axhline(0, color='black', lw=0.8, ls='--', zorder=1)
        ax_delta.scatter(jit2, deltas, color=color, s=DOT_SIZE, alpha=0.75, zorder=3)

        ax_delta.text(0.5, 0.97, f'{s}  p={p:.2e}',
                      ha='center', va='top', fontsize=6.5,
                      transform=ax_delta.transAxes)
        ax_delta.text(0.5, 0.02, f'n={len(sub)}',
                      ha='center', va='bottom', fontsize=6.5, color='grey',
                      transform=ax_delta.transAxes)

        ax_delta.set_xlim(-0.5, 0.5)
        ax_delta.set_xticks([])
        ax_delta.tick_params(axis='y', labelsize=7)
        ax_delta.set_ylabel('ΔCIITA\n(S100P+ − S100P-)', fontsize=7)
        ax_delta.spines[['top', 'right', 'bottom']].set_visible(False)
        ax_delta.yaxis.set_ticks_position('left')
        ax_delta.yaxis.set_label_position('left')

    # share y limits across both paired dot plots
    all_vals = pd.concat([
        wide['CIITA_mean_S100Pneg'].dropna(),
        wide['CIITA_mean_S100Ppos'].dropna()
    ])
    ymin_shared = all_vals.min() * 0.9
    ymax_shared = all_vals.max() * 1.1
    ax_slots[0].set_ylim(ymin_shared, ymax_shared)
    ax_slots[2].set_ylim(ymin_shared, ymax_shared)
    # share delta ylims too
    all_deltas = wide['delta_pos_minus_neg'].dropna()
    delta_ymin = all_deltas.min() - np.ptp(all_deltas.values) * 0.08
    delta_ymax = all_deltas.max() + np.ptp(all_deltas.values) * 0.45
    ax_slots[1].set_ylim(delta_ymin, delta_ymax)
    ax_slots[3].set_ylim(delta_ymin, delta_ymax)

    fig.suptitle('CIITA expression in S100P+ vs S100P- epithelial cells',
                 fontsize=9, y=1.05)
    return fig

def sig_label(p):
    """
    Convert a p-value to a significance star string.

    Parameters
    ----------
    p : float

    Returns
    -------
    str : '***', '**', '*', or 'ns'
    """
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    return 'ns'

In [ ]:
#from ceiba.plot_utils import plot_ciita_s100p_paired

fig = plot_ciita_s100p_paired(wide, figsize=(5,5))
fig.savefig(fig_out / 'fig3c_ciita_s100p_stratified.pdf', bbox_inches='tight')
plt.show()